# 03 — Classical Repair Hooks

On real hardware most raw samples violate constraints. A repair heuristic turns those wasted shots into feasible candidates. The solver calls yours automatically on every infeasible sample.

Below we use the `MockBackend` (pure noise) to see repair doing *all* the work — the harshest possible test of your heuristic.

In [ ]:
import sys; sys.path.insert(0, "../..")
from problems.vrp import VehicleRoutingProblem
from qlkit import LogisticsSolver, MockBackend, GreedyAssignmentRepair

problem = VehicleRoutingProblem.small_instance()

for repair in (None, GreedyAssignmentRepair()):
    with LogisticsSolver(problem, MockBackend(seed=0), repair=repair) as solver:
        result = solver.solve({"one_hot": 12.0, "capacity": 2.0}, shots=512)
        label = type(repair).__name__ if repair else "no repair"
        print(f"{label:24s} feasible={result.samples.feasible_fraction:6.1%}  "
              f"best cost={result.metrics.true_objective:.2f}")

In [ ]:
# Pre/post-repair pairs are preserved so you can measure your heuristic:
with LogisticsSolver(problem, MockBackend(seed=0), repair=GreedyAssignmentRepair()) as solver:
    result = solver.solve({"one_hot": 12.0, "capacity": 2.0}, shots=512)
record = next(r for r in result.samples.records if r.pre_repair)
print("before:", record.pre_repair.bits)
print("after: ", record.solution.bits, "-> feasible:", record.hard_feasible)

## Your turn

Open `starter_kit/my_repair.py`. The contract: return a solution **at least as feasible** as the input, never mutate the input. Ideas: repair toward the objective, not just feasibility; add a feasibility-preserving local-search pass; invent problem-specific moves (customer swaps).